# OLMoE Training on Kaggle

Fine-tune / pre-train an OLMoE (Mixture-of-Experts) model on Kaggle GPUs using pre-tokenized `.npy` shards.

**Author:** Tristan Martin & Iliass Lasri | **Date:** 2025

**Before running — checklist:**
- Kaggle > Settings > Accelerator: `GPU T4 x2` or `P100`
- Kaggle > Settings > Internet: **ON**
- Kaggle > Add-ons > Secrets: add secret named `WANDB_API_KEY` *(optional — runs offline without it)*
- Kaggle > Input: attach the `olmoe-dataset` dataset (contains `.npy` files)
- Edit the Configuration cell below to set hyperparameters (data paths are auto-detected)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# All user-facing variables live here. Edit as needed.

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_GLOB_PATTERN    = "/kaggle/input/**/*.npy"  # auto-finds .npy files in any attached dataset
REPO_URL             = "https://github.com/iliasslasri/OLMoE.git"
OLMOE_DIR            = "/kaggle/working/OLMoE"                     # clone destination
PIP_OVERRIDE_DIR     = "/kaggle/working/_pip_overrides"            # isolated pip installs
TOKENIZER_URL        = "https://huggingface.co/allenai/gpt-neox-olmo-dolma-v1_5/resolve/main/tokenizer.json"
TOKENIZER_LOCAL_PATH = "tokenizers/allenai_gpt-neox-olmo-dolma-v1_5.json"
SAVE_FOLDER          = "/kaggle/working/runs/${run_name}"          # checkpoint output

# ── OLMo submodule (points to Tristan22400/OLMo fork) ─────────────────────────
OLMO_REPO_URL        = "https://github.com/Tristan22400/OLMo.git"
OLMO_BRANCH          = "routing/moe-strategies"

# ── Megablocks fork (routing strategies) ───────────────────────────────────────
MEGABLOCKS_PIP_URL   = "git+https://github.com/Tristan22400/megablocks.git@routing/auxiliary-loss-free"

# ── Training hyperparameters ───────────────────────────────────────────────────
GLOBAL_TRAIN_BATCH_SIZE    = 32          # total tokens per optimizer step across all GPUs
DEVICE_TRAIN_MICROBATCH_SIZE = 4         # per-GPU per-step (controls peak VRAM)
MAX_SEQUENCE_LENGTH        = 2048        # context window
MAX_TRAINING_STEPS         = 500         # stop after this many optimizer steps (int or "2ep" for epochs)
PRECISION                  = "amp_fp16"  # mixed-precision mode
ACTIVATION_CHECKPOINTING   = "fine_grained"  # fine_grained | null (see Cell 7 comments)
MOE_DROPLESS               = False       # dropless MoE requires megablocks sparse kernels
MOE_MLP_IMPL               = "sparse"    # sparse | dense

# ── MoE Routing Strategy ──────────────────────────────────────────────────────
# "learned"    → standard auxiliary load balancing loss (baseline)
# "loss_free"  → auxiliary-loss-free additive bias (Wang et al., 2024)
# "random"     → uniform random assignment (ablation baseline)
MOE_ROUTING_TYPE           = "loss_free"

# ── W&B ────────────────────────────────────────────────────────────────────────
WANDB_ENTITY_NAME  = "iliass-lasri-team"
WANDB_PROJECT_NAME = "olmoe-1"
WANDB_RUN_MODE     = "online"            # online | offline | disabled

# ── System ─────────────────────────────────────────────────────────────────────
OMP_NUM_THREADS_TRAIN = "4"              # OpenMP threads during training
MIN_DISK_FREE_GB      = 8                # warn if less than this available

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import glob
import importlib
import os
import pathlib
import re
import shutil
import socket
import subprocess
import sys

import torch
import yaml

## 1. System Check

In [ ]:
if not shutil.which("nvidia-smi"):
    raise EnvironmentError(
        "\n'nvidia-smi' not found — no GPU attached.\n"
        "Fix: Kaggle > Settings > Accelerator > GPU T4 x2 or P100 > Save"
    )
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
N_GPU = torch.cuda.device_count()
print(f"GPUs    : {N_GPU}")
for i in range(N_GPU):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
assert N_GPU > 0, "No GPU detected. Enable a GPU accelerator in Kaggle settings."

In [ ]:
disk_usage_output = subprocess.run(
    ['df', '-h', '/kaggle/working'], capture_output=True, text=True
).stdout
print(disk_usage_output)

free_gb = float(subprocess.run(
    ['df', '--output=avail', '-BG', '/kaggle/working'],
    capture_output=True, text=True
).stdout.strip().split()[-1].replace('G', ''))

if free_gb < MIN_DISK_FREE_GB:
    print(f"WARNING: only {free_gb:.1f} GB free — may be tight.")
else:
    print(f"Disk OK: {free_gb:.1f} GB free.")

## 2. Environment Setup

In [ ]:
# ── Resolve tokenized data paths ──────────────────────────────────────────────
TOKENIZED_DATA_PATHS = sorted(glob.glob(DATA_GLOB_PATTERN))

assert len(TOKENIZED_DATA_PATHS) > 0, (
    f"\nNo .npy files found at {DATA_GLOB_PATTERN}.\n"
    "Possible fixes:\n"
    "  1. Check the dataset is attached in Kaggle > Input\n"
    "  2. Run: !find /kaggle/input -name '*.npy' to see actual paths\n"
    "  3. Update DATA_GLOB_PATTERN in the Configuration cell."
)

print(f"Found {len(TOKENIZED_DATA_PATHS)} tokenized shard(s):")
for p in TOKENIZED_DATA_PATHS:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {p}  ({size_mb:.0f} MB)")

In [ ]:
# ── W&B credentials ───────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_ENTITY"]  = WANDB_ENTITY_NAME
os.environ["WANDB_PROJECT"] = WANDB_PROJECT_NAME
os.environ["WANDB_MODE"]    = WANDB_RUN_MODE
print("W&B configured.")

In [ ]:
# ── Clone repo + submodule ────────────────────────────────────────────────────
if not os.path.exists(OLMOE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, OLMOE_DIR], check=True)
    # Override submodule URL to use our OLMo fork (Kaggle has no SSH keys)
    subprocess.run(
        ["git", "config", "submodule.OLMo.url", OLMO_REPO_URL],
        cwd=OLMOE_DIR, check=True,
    )
    # Full clone (not --depth 1) so the branch is reachable
    subprocess.run(["git", "submodule", "update", "--init"], cwd=OLMOE_DIR, check=True)
    # Fetch and checkout the routing/moe-strategies branch with our config+train changes
    subprocess.run(["git", "fetch", "origin", OLMO_BRANCH],
                   cwd=f"{OLMOE_DIR}/OLMo", check=True)
    subprocess.run(["git", "checkout", OLMO_BRANCH],
                   cwd=f"{OLMOE_DIR}/OLMo", check=True)
else:
    print("Repo already present, skipping clone.")

os.chdir(OLMOE_DIR)
print("cwd:", os.getcwd())
print("main:", subprocess.run(["git", "log", "--oneline", "-1"],
                               capture_output=True, text=True).stdout.strip())
print("OLMo:", subprocess.run(["git", "log", "--oneline", "-1"],
                               capture_output=True, text=True, cwd="OLMo").stdout.strip())
print("OLMo branch:", subprocess.run(["git", "branch", "--show-current"],
                               capture_output=True, text=True, cwd="OLMo").stdout.strip())

In [ ]:
# ── Dependency installer helpers ──────────────────────────────────────────────
#
# Why this complexity? See root causes:
#   - ai2-olmo-core==0.1.0 pins huggingface_hub ~0.36.x; transformers needs >=1.3.0
#   - OLMo[train] may downgrade torch (conflicts with Kaggle's 2.9.x)
#   - megablocks must compile against the post-downgrade torch ABI
#   - Kaggle's system pyarrow/pandas break if numpy downgrades in-kernel

OLMO_SRC = f"{OLMOE_DIR}/OLMo"
os.makedirs(PIP_OVERRIDE_DIR, exist_ok=True)


def pip_install(*args):
    """Install packages quietly via pip, raising on failure."""
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        raise RuntimeError(f"pip failed: {' '.join(args)}")


def pip_install_isolated(*args):
    """Install packages (--no-deps) into PIP_OVERRIDE_DIR to shadow system copies."""
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", PIP_OVERRIDE_DIR, "--no-deps"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        raise RuntimeError(f"pip_install_isolated failed: {' '.join(args)}")

In [ ]:
def purge_cached_modules(*prefixes):
    """Remove cached modules from sys.modules to force reimport."""
    stale = [k for k in sys.modules
             if any(k == p or k.startswith(p + ".") for p in prefixes)]
    for k in stale:
        del sys.modules[k]
    if stale:
        print(f"  Purged {len(stale)} cached module(s): {prefixes}")


# ── Step 1: OLMo with training extras ─────────────────────────────────────────
print("[1/5] Installing OLMo[train]...")
pip_install("-e", "OLMo[train]")

if OLMO_SRC not in sys.path:
    sys.path.insert(1, OLMO_SRC)
    print(f"  Added {OLMO_SRC} to sys.path")

# ── Step 2: megablocks (our fork with routing strategies) ──────────────────────
print("[2/5] Installing megablocks (routing strategies fork, force-recompile)...")
print(f"  Source: {MEGABLOCKS_PIP_URL}")
pip_install("--force-reinstall", MEGABLOCKS_PIP_URL)

In [ ]:
# ── Step 3: Reinstall torchvision matching the (possibly downgraded) torch ────
print("[3/5] Detecting torch version after OLMo install...")
detect = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
)
installed_torch_version = detect.stdout.strip()
cuda_tag = installed_torch_version.split("+")[1] if "+" in installed_torch_version else "cpu"
print(f"  Detected torch=={installed_torch_version}, CUDA tag: {cuda_tag}")

print("[4/5] Reinstalling torchvision for detected torch...")
pip_install("--force-reinstall", "torchvision",
            "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}")
purge_cached_modules("torchvision")

# ── Step 4: Force-upgrade huggingface_hub ─────────────────────────────────────
print("[5/5] Fixing huggingface_hub / tokenizers versions...")
pip_install_isolated("huggingface_hub>=1.3.0,<2.0")
pip_install_isolated("tokenizers>=0.22.0,<=0.23.0")

if PIP_OVERRIDE_DIR not in sys.path:
    sys.path.insert(0, PIP_OVERRIDE_DIR)
    print(f"  Inserted {PIP_OVERRIDE_DIR} at sys.path[0]")

purge_cached_modules("huggingface_hub", "tokenizers")

In [ ]:
# ── Verify dependencies ───────────────────────────────────────────────────────
# NOTE: megablocks is NOT verified here. Its .so was compiled against the
# downgraded torch, but the kernel still has the old torch in memory.
# The training subprocess loads it correctly from a fresh interpreter.
print("Verifying imports...")
try:
    from huggingface_hub import is_offline_mode  # noqa: F401
    print("  ✓ huggingface_hub version OK")
except ImportError as e:
    raise RuntimeError(f"huggingface_hub fix failed: {e}") from e

for mod in ["omegaconf", "wandb"]:
    importlib.import_module(mod)
    print(f"  ✓ {mod}")

import huggingface_hub
import tokenizers as hf_tok
print(f"  huggingface_hub=={huggingface_hub.__version__}")
print(f"  tokenizers=={hf_tok.__version__}")
print(f"  torch (subprocess)=={installed_torch_version}")
print("All dependencies OK. megablocks verified by dry-run subprocess.")

In [ ]:
# ── Download tokenizer ────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(TOKENIZER_LOCAL_PATH), exist_ok=True)

if not os.path.exists(TOKENIZER_LOCAL_PATH):
    print("Downloading tokenizer...")
    result = subprocess.run(
        ["wget", "-q", TOKENIZER_URL, "-O", TOKENIZER_LOCAL_PATH],
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not os.path.exists(TOKENIZER_LOCAL_PATH):
        raise RuntimeError(
            "Tokenizer download failed. Check internet is enabled "
            "(Kaggle > Settings > Internet ON).\n" + result.stderr
        )
    size_kb = os.path.getsize(TOKENIZER_LOCAL_PATH) / 1024
    print(f"  Saved to {TOKENIZER_LOCAL_PATH} ({size_kb:.0f} KB)")
    assert size_kb > 100, f"File too small ({size_kb:.0f} KB) — download may have failed."
else:
    print(f"Tokenizer already present at {TOKENIZER_LOCAL_PATH}")

## 3. Training Configuration

In [ ]:
# ── Pre-flight batch-size checks ──────────────────────────────────────────────
if GLOBAL_TRAIN_BATCH_SIZE % N_GPU != 0:
    raise ValueError(
        f"GLOBAL_TRAIN_BATCH_SIZE ({GLOBAL_TRAIN_BATCH_SIZE}) "
        f"must be divisible by N_GPU ({N_GPU})"
    )
device_batch = GLOBAL_TRAIN_BATCH_SIZE // N_GPU
if device_batch % DEVICE_TRAIN_MICROBATCH_SIZE != 0:
    raise ValueError(
        f"device_batch ({device_batch}) must be divisible by "
        f"DEVICE_TRAIN_MICROBATCH_SIZE ({DEVICE_TRAIN_MICROBATCH_SIZE})"
    )
grad_accum_steps = device_batch // DEVICE_TRAIN_MICROBATCH_SIZE
print(f"Batch: global={GLOBAL_TRAIN_BATCH_SIZE}, per-GPU={device_batch}, "
      f"microbatch={DEVICE_TRAIN_MICROBATCH_SIZE}, grad_accum={grad_accum_steps}")

In [ ]:
# ── Patch YAML config ─────────────────────────────────────────────────────────
# Activation checkpointing notes for MoE (block_type: moe):
#   fine_grained → checkpoints attn+norms per block, skips MoE FFN (~20% overhead)
#   null         → no checkpointing (highest memory usage)
#   one_in_two / one_in_four / whole_layer → DENSE-only, raises error with MoE

CONFIG_PATH = pathlib.Path(OLMOE_DIR) / "configs/olmoe-small.yml"
assert CONFIG_PATH.exists(), f"Config not found: {CONFIG_PATH}"

cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg["data"]["paths"]                 = list(TOKENIZED_DATA_PATHS)
cfg["model"]["moe_dropless"]         = MOE_DROPLESS
cfg["model"]["moe_mlp_impl"]         = MOE_MLP_IMPL
cfg["model"]["max_sequence_length"]  = MAX_SEQUENCE_LENGTH
cfg["model"]["moe_routing_type"]     = MOE_ROUTING_TYPE
cfg["max_duration"]                  = MAX_TRAINING_STEPS
cfg["precision"]                     = PRECISION
cfg["activation_checkpointing"]      = ACTIVATION_CHECKPOINTING
cfg["global_train_batch_size"]       = GLOBAL_TRAIN_BATCH_SIZE
cfg["device_train_microbatch_size"]  = DEVICE_TRAIN_MICROBATCH_SIZE
cfg["save_folder"]                   = SAVE_FOLDER
cfg["compile"]                       = None
cfg["fsdp"].pop("use_orig_params", None)

# For loss_free routing, ensure z-loss is active and LB loss weight is set
# (LB loss weight is needed for stats computation even though it's not added
# to the training objective).
if MOE_ROUTING_TYPE == "loss_free":
    cfg["model"].setdefault("moe_zloss_weight", 0.001)

config_yaml_text = yaml.dump(cfg, default_flow_style=False, sort_keys=False)
config_yaml_text = re.sub(r"'(\$\{[^}]+\}[^']*)', ", r"\1", config_yaml_text)
CONFIG_PATH.write_text(config_yaml_text)
print(f"Config written to {CONFIG_PATH}")
print(f"  moe_routing_type: {MOE_ROUTING_TYPE}")

In [ ]:
# ── Verify patched config ─────────────────────────────────────────────────────
verified_config = yaml.safe_load(CONFIG_PATH.read_text())

config_checks = [
    (verified_config["data"]["paths"] == list(TOKENIZED_DATA_PATHS), "data.paths mismatch"),
    (verified_config["model"]["max_sequence_length"] == MAX_SEQUENCE_LENGTH, "max_sequence_length"),
    (verified_config["model"]["moe_dropless"] == MOE_DROPLESS, "moe_dropless"),
    (verified_config["model"]["moe_routing_type"] == MOE_ROUTING_TYPE, "moe_routing_type"),
    (verified_config["activation_checkpointing"] == ACTIVATION_CHECKPOINTING, "activation_checkpointing"),
    (verified_config["global_train_batch_size"] == GLOBAL_TRAIN_BATCH_SIZE, "global_train_batch_size"),
]
errors = [msg for ok, msg in config_checks if not ok]
if errors:
    raise RuntimeError("Config verification FAILED:\n  " + "\n  ".join(errors))

In [ ]:
CHECKPOINTING_DESCRIPTIONS = {
    "fine_grained": "attn+norms only, skips MoE FFN (~20% compute overhead)",
    "null": "none (highest memory usage)",
}
ROUTING_DESCRIPTIONS = {
    "learned": "standard auxiliary load balancing loss",
    "loss_free": "auxiliary-loss-free additive bias (Wang et al., 2024)",
    "random": "uniform random assignment (ablation baseline)",
}
ckpt_str = verified_config["activation_checkpointing"]
routing_str = verified_config["model"]["moe_routing_type"]
print("=== Config OK ===")
print(f"  max_duration         : {verified_config['max_duration']}")
print(f"  precision            : {verified_config['precision']}")
print(f"  max_sequence_length  : {verified_config['model']['max_sequence_length']}")
print(f"  moe_dropless         : {verified_config['model']['moe_dropless']}")
print(f"  moe_mlp_impl         : {verified_config['model']['moe_mlp_impl']}")
print(f"  moe_routing_type     : {routing_str}  ({ROUTING_DESCRIPTIONS.get(routing_str, '?')})")
print(f"  activation_ckpt      : {ckpt_str}  ({CHECKPOINTING_DESCRIPTIONS.get(str(ckpt_str), '?')})")
print(f"  compile              : {verified_config.get('compile')}")
print(f"  global_batch_size    : {verified_config['global_train_batch_size']}")
print(f"  microbatch_size      : {verified_config['device_train_microbatch_size']}")
print(f"  data.paths ({len(verified_config['data']['paths'])} file(s)):")
for p in verified_config["data"]["paths"]:
    print(f"    {p}")

In [ ]:
# ── Dry-run: import + config validation (no GPU needed) ───────────────────────
# Runs train.py without torchrun. Expected exit: ValueError at
# dist.init_process_group() — means all imports + config checks passed.

dry_run_env = os.environ.copy()
dry_run_env["OLMO_TASK"]       = "model"
dry_run_env["PYTHONPATH"]      = f"{PIP_OVERRIDE_DIR}:{OLMO_SRC}:" + dry_run_env.get("PYTHONPATH", "")
dry_run_env["OMP_NUM_THREADS"] = "1"

result = subprocess.run(
    [sys.executable, f"{OLMOE_DIR}/OLMo/scripts/train.py", str(CONFIG_PATH)],
    capture_output=True, text=True, env=dry_run_env, cwd=OLMOE_DIR,
)

print(f"Dry run exit code: {result.returncode}")
print("--- STDOUT ---")
print(result.stdout[:2000] or "(no stdout)")
print("--- STDERR (last 2000 chars) ---")
print(result.stderr[-2000:] or "(no stderr)")

In [ ]:
# ── Classify dry-run exit ─────────────────────────────────────────────────────
IMPORT_ERROR_MARKERS = ["ModuleNotFoundError", "ImportError", "cannot import name"]
CONFIG_ERROR_MARKERS = [
    "OmegaConf", "yaml", "KeyError", "missing mandatory value",
    "ConfigAttributeError", "MissingMandatoryValue",
]

has_import_error = any(m in result.stderr for m in IMPORT_ERROR_MARKERS)
has_config_error = any(m in result.stderr for m in CONFIG_ERROR_MARKERS)
reached_distributed = "RANK expected, but not set" in result.stderr

if has_import_error:
    raise RuntimeError("Import error in dry run — fix dependencies before training.")
if has_config_error:
    raise RuntimeError("Config error in dry run — check config patches.")
if reached_distributed:
    print("\n✓ Dry run passed: all imports and config loaded OK.")
    print("  (Script stopped at dist.init_process_group — expected without torchrun.)")
elif result.returncode == 0:
    print("\n✓ Dry run passed cleanly.")
else:
    raise RuntimeError(f"Unexpected failure (exit {result.returncode}). See STDERR above.")

## 4. Launch Training

In [ ]:
def find_free_port():
    """Find and return a free TCP port on localhost."""
    with socket.socket() as s:
        s.bind(('', 0))
        return s.getsockname()[1]


rdzv_port = find_free_port()
print(f"Using rdzv port: {rdzv_port}")
print(f"Launching on {N_GPU} GPU(s)...")

train_env = os.environ.copy()
train_env["OLMO_TASK"]              = "model"
train_env["PYTHONPATH"]             = f"{PIP_OVERRIDE_DIR}:{OLMO_SRC}:" + train_env.get("PYTHONPATH", "")
train_env["OMP_NUM_THREADS"]        = OMP_NUM_THREADS_TRAIN
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

train_cmd = [
    sys.executable, "-m", "torch.distributed.run",
    f"--nproc-per-node={N_GPU}",
    "--nnodes=1", "--node_rank=0",
    "--rdzv_backend=c10d",
    f"--rdzv_endpoint=localhost:{rdzv_port}",
    "OLMo/scripts/train.py",
    str(CONFIG_PATH),
]
print("Command:", " ".join(train_cmd))
print("-" * 60)

In [ ]:
# ── Run training (streams output live) ────────────────────────────────────────
process = subprocess.Popen(
    train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=train_env,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Training failed (exit code {process.returncode})")
print("\nTraining completed successfully.")